# Objective 3 — Step 11: German Cost-Sensitive Threshold Development

This notebook develops the **decision-threshold component on German Credit only** after the hybrid architecture and sigmoid calibration have already been frozen.

## Frozen upstream framework
- Group-aware Chi-Square Top-75%
- Balanced Logistic Regression
- Balanced Random Forest
- Balanced XGBoost
- Equal-probability Soft Voting
- Sigmoid probability calibration

## Threshold strategies evaluated
1. Fixed 0.50
2. Max-MCC threshold
3. Max-F1 threshold
4. Max-Balanced-Accuracy threshold
5. Minimum cost for FN:FP = 2:1
6. Minimum cost for FN:FP = 5:1
7. Minimum cost for FN:FP = 10:1

The cost ratios are **sensitivity scenarios**, not claims about actual bank-specific monetary costs.

## Leakage control
Thresholds are selected only from the corresponding outer-training partition.

To avoid choosing a threshold on probabilities produced by a calibrator that was fitted to those exact same rows, the notebook creates **cross-fitted calibrated OOF probabilities**:

- Step-9 raw hybrid OOF probabilities are taken from the outer-training partition.
- A second 5-fold grouped split fits the sigmoid calibrator on four folds and calibrates the held-out fold.
- These cross-fitted calibrated OOF probabilities are used to choose the threshold.
- The resulting threshold is applied to the untouched Step-9 calibrated outer-test probabilities.

Australian and Taiwan are not used in threshold-rule selection.


In [1]:
%pip install pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2
[notice] To update, run: C:\Users\hp\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [2]:

from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings("ignore")

BASE_DIR = Path(r"D:\PHD\Research Paper writing\3rd Obj. paper")

STEP9_DIR = BASE_DIR / "results" / "calibration_german_development"
DATA_DIR = BASE_DIR / "data" / "processed"

OUT_DIR = BASE_DIR / "results" / "threshold_german_development"
OUT_DIR.mkdir(parents=True, exist_ok=True)

GERMAN_FILE = DATA_DIR / "german_credit_cleaned.csv"
OOF_FILE = STEP9_DIR / "german_calibration_inner_oof_probabilities.csv"
OUTER_PREDICTIONS_FILE = STEP9_DIR / "german_calibration_outer_predictions.csv"
CALIBRATOR_FILE = STEP9_DIR / "german_calibrator_parameters.csv"

required = [
    GERMAN_FILE,
    OOF_FILE,
    OUTER_PREDICTIONS_FILE,
    CALIBRATOR_FILE,
]

missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Step-9 outputs are incomplete:\n" + "\n".join(missing)
    )

REPEAT_SEEDS = [42, 142, 242, 342, 442]
OUTER_FOLDS = 5
CALIBRATION_CROSSFIT_FOLDS = 5
EPS = 1e-6

THRESHOLD_STRATEGIES = [
    "Fixed0.50",
    "MaxMCC",
    "MaxF1",
    "MaxBalancedAccuracy",
    "MinCost_FN2_FP1",
    "MinCost_FN5_FP1",
    "MinCost_FN10_FP1",
]

print("Output folder:", OUT_DIR)


Output folder: D:\PHD\Research Paper writing\3rd Obj. paper\results\threshold_german_development


## 1. Load Step-9 probabilities and German data

In [3]:

df = pd.read_csv(GERMAN_FILE)
inner_oof = pd.read_csv(OOF_FILE)
outer_predictions = pd.read_csv(OUTER_PREDICTIONS_FILE)
calibrator_parameters = pd.read_csv(CALIBRATOR_FILE)

y = df["adverse_target"].astype(int).copy()
groups = df["profile_group_id"].astype(str).copy()

print("German records:", len(df))
print("Step-9 inner OOF rows:", len(inner_oof))
print("Step-9 outer prediction rows:", len(outer_predictions))
print("Calibrators:", len(calibrator_parameters))

assert len(df) == 1000
assert len(inner_oof) == 25 * 800
assert len(outer_predictions) == 5000
assert len(calibrator_parameters) == 25


German records: 1000
Step-9 inner OOF rows: 20000
Step-9 outer prediction rows: 5000
Calibrators: 25


## 2. Recreate outer folds

In [4]:

outer_splits = {}

dummy_X = np.zeros((len(df), 1))

for repeat_no, seed in enumerate(REPEAT_SEEDS, start=1):
    splitter = StratifiedGroupKFold(
        n_splits=OUTER_FOLDS,
        shuffle=True,
        random_state=seed,
    )

    for fold_no, (train_idx, test_idx) in enumerate(
        splitter.split(dummy_X, y, groups),
        start=1,
    ):
        run_id = f"R{repeat_no}_F{fold_no}"

        outer_splits[run_id] = {
            "repeat": repeat_no,
            "fold": fold_no,
            "seed": seed,
            "train_idx": np.asarray(train_idx, dtype=int),
            "test_idx": np.asarray(test_idx, dtype=int),
        }

print("Outer runs:", len(outer_splits))
assert len(outer_splits) == 25


Outer runs: 25


## 3. Sigmoid calibration helper

In [5]:

def clip_probability(p):
    return np.clip(
        np.asarray(p, dtype=float),
        EPS,
        1.0 - EPS,
    )


def logit(p):
    p = clip_probability(p)
    return np.log(p / (1.0 - p))


def fit_sigmoid(raw_probability, y_true):
    model = LogisticRegression(
        C=1e6,
        solver="lbfgs",
        max_iter=5000,
        random_state=42,
    )

    model.fit(
        logit(raw_probability).reshape(-1, 1),
        np.asarray(y_true, dtype=int),
    )

    return model


def apply_sigmoid(model, raw_probability):
    return model.predict_proba(
        logit(raw_probability).reshape(-1, 1)
    )[:, 1]


## 4. Create cross-fitted calibrated OOF probabilities

In [6]:

crossfit_rows = []

for run_id, info in outer_splits.items():
    train_idx = info["train_idx"]

    y_outer_train = (
        y.iloc[train_idx]
        .reset_index(drop=True)
    )

    groups_outer_train = (
        groups.iloc[train_idx]
        .reset_index(drop=True)
    )

    run_oof = (
        inner_oof[
            inner_oof["run_id"] == run_id
        ]
        .sort_values("outer_train_position")
        .reset_index(drop=True)
    )

    assert len(run_oof) == len(train_idx)
    assert np.array_equal(
        run_oof["outer_train_position"].to_numpy(),
        np.arange(len(train_idx)),
    )

    assert np.array_equal(
        run_oof["y_true"].astype(int).to_numpy(),
        y_outer_train.to_numpy(),
    )

    raw_oof_probability = (
        run_oof["raw_oof_hybrid_probability"]
        .to_numpy(dtype=float)
    )

    calibrated_crossfit = np.full(
        len(run_oof),
        np.nan,
        dtype=float,
    )

    calibration_splitter = StratifiedGroupKFold(
        n_splits=CALIBRATION_CROSSFIT_FOLDS,
        shuffle=True,
        random_state=40000 + info["seed"] + info["fold"],
    )

    dummy = np.zeros((len(run_oof), 1))

    for calibration_fold, (
        calibration_train_pos,
        calibration_valid_pos,
    ) in enumerate(
        calibration_splitter.split(
            dummy,
            y_outer_train,
            groups_outer_train,
        ),
        start=1,
    ):
        calibrator = fit_sigmoid(
            raw_oof_probability[calibration_train_pos],
            y_outer_train.iloc[
                calibration_train_pos
            ].to_numpy(),
        )

        calibrated_values = apply_sigmoid(
            calibrator,
            raw_oof_probability[calibration_valid_pos],
        )

        calibrated_crossfit[
            calibration_valid_pos
        ] = calibrated_values

        for local_pos, train_position in enumerate(
            calibration_valid_pos
        ):
            crossfit_rows.append({
                "run_id": run_id,
                "repeat": info["repeat"],
                "fold": info["fold"],
                "calibration_crossfit_fold": calibration_fold,
                "outer_train_position": int(train_position),
                "y_true": int(
                    y_outer_train.iloc[train_position]
                ),
                "raw_oof_probability": float(
                    raw_oof_probability[train_position]
                ),
                "crossfitted_sigmoid_probability": float(
                    calibrated_values[local_pos]
                ),
            })

    assert np.isfinite(calibrated_crossfit).all()


crossfit_oof = pd.DataFrame(crossfit_rows)

crossfit_oof.to_csv(
    OUT_DIR / "german_threshold_crossfitted_calibrated_oof.csv",
    index=False,
)

print("Cross-fitted calibrated OOF rows:", len(crossfit_oof))
assert len(crossfit_oof) == 25 * 800


Cross-fitted calibrated OOF rows: 20000


## 5. Threshold-search metrics

In [7]:

def threshold_metrics(y_true, probability, threshold):
    y_true = np.asarray(y_true, dtype=int)
    probability = np.asarray(probability, dtype=float)

    prediction = (
        probability >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        prediction,
        labels=[0, 1],
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    gmean = (
        math.sqrt(specificity * sensitivity)
        if not np.isnan(
            specificity + sensitivity
        )
        else np.nan
    )

    return {
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_true, prediction),
        "precision_adverse": precision_score(
            y_true,
            prediction,
            pos_label=1,
            zero_division=0,
        ),
        "recall_adverse": recall_score(
            y_true,
            prediction,
            pos_label=1,
            zero_division=0,
        ),
        "specificity": specificity,
        "f1_adverse": f1_score(
            y_true,
            prediction,
            pos_label=1,
            zero_division=0,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            prediction,
        ),
        "mcc": matthews_corrcoef(
            y_true,
            prediction,
        ),
        "gmean": gmean,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "cost_FN1_FP1": int(fp + fn),
        "cost_FN2_FP1": int(fp + 2 * fn),
        "cost_FN5_FP1": int(fp + 5 * fn),
        "cost_FN10_FP1": int(fp + 10 * fn),
    }


def candidate_thresholds(probability):
    values = np.unique(
        clip_probability(probability)
    )

    if len(values) == 1:
        return np.array([0.5])

    midpoints = (
        values[:-1]
        + values[1:]
    ) / 2.0

    candidates = np.unique(
        np.concatenate([
            np.array([EPS, 0.5, 1.0 - EPS]),
            midpoints,
        ])
    )

    return candidates


def choose_threshold(
    y_true,
    probability,
    strategy,
):
    if strategy == "Fixed0.50":
        return 0.50

    rows = [
        threshold_metrics(
            y_true,
            probability,
            threshold,
        )
        for threshold in candidate_thresholds(
            probability
        )
    ]

    table = pd.DataFrame(rows)

    if strategy == "MaxMCC":
        ranked = table.sort_values(
            [
                "mcc",
                "balanced_accuracy",
                "f1_adverse",
            ],
            ascending=[
                False,
                False,
                False,
            ],
        )

    elif strategy == "MaxF1":
        ranked = table.sort_values(
            [
                "f1_adverse",
                "mcc",
                "balanced_accuracy",
            ],
            ascending=[
                False,
                False,
                False,
            ],
        )

    elif strategy == "MaxBalancedAccuracy":
        ranked = table.sort_values(
            [
                "balanced_accuracy",
                "mcc",
                "f1_adverse",
            ],
            ascending=[
                False,
                False,
                False,
            ],
        )

    elif strategy.startswith("MinCost_FN"):
        fn_cost = int(
            strategy
            .split("MinCost_FN")[1]
            .split("_FP1")[0]
        )

        cost_column = (
            f"cost_FN{fn_cost}_FP1"
        )

        ranked = table.sort_values(
            [
                cost_column,
                "mcc",
                "balanced_accuracy",
            ],
            ascending=[
                True,
                False,
                False,
            ],
        )

    else:
        raise ValueError(strategy)

    best = ranked.iloc[0]

    return float(best["threshold"])


## 6. Choose thresholds from cross-fitted calibrated OOF and evaluate untouched outer-test probabilities

In [8]:

threshold_rows = []
outer_result_rows = []

for run_id, info in outer_splits.items():
    run_crossfit = (
        crossfit_oof[
            crossfit_oof["run_id"] == run_id
        ]
        .sort_values("outer_train_position")
    )

    y_train_oof = (
        run_crossfit["y_true"]
        .astype(int)
        .to_numpy()
    )

    calibrated_train_oof = (
        run_crossfit[
            "crossfitted_sigmoid_probability"
        ]
        .to_numpy(dtype=float)
    )

    run_outer = outer_predictions[
        outer_predictions["run_id"] == run_id
    ].copy()

    y_outer_test = (
        run_outer["y_true"]
        .astype(int)
        .to_numpy()
    )

    calibrated_outer_test = (
        run_outer["sigmoid_probability"]
        .to_numpy(dtype=float)
    )

    for strategy in THRESHOLD_STRATEGIES:
        threshold = choose_threshold(
            y_train_oof,
            calibrated_train_oof,
            strategy,
        )

        train_metrics = threshold_metrics(
            y_train_oof,
            calibrated_train_oof,
            threshold,
        )

        test_metrics = threshold_metrics(
            y_outer_test,
            calibrated_outer_test,
            threshold,
        )

        threshold_rows.append({
            "run_id": run_id,
            "repeat": info["repeat"],
            "fold": info["fold"],
            "threshold_strategy": strategy,
            "selected_threshold": threshold,
            "training_oof_mcc": train_metrics["mcc"],
            "training_oof_f1": train_metrics["f1_adverse"],
            "training_oof_balanced_accuracy": (
                train_metrics["balanced_accuracy"]
            ),
            "training_oof_recall": (
                train_metrics["recall_adverse"]
            ),
            "training_oof_precision": (
                train_metrics["precision_adverse"]
            ),
            "training_oof_cost_FN2_FP1": (
                train_metrics["cost_FN2_FP1"]
            ),
            "training_oof_cost_FN5_FP1": (
                train_metrics["cost_FN5_FP1"]
            ),
            "training_oof_cost_FN10_FP1": (
                train_metrics["cost_FN10_FP1"]
            ),
        })

        n_test = len(y_outer_test)

        outer_result_rows.append({
            "dataset": "German Credit",
            "run_id": run_id,
            "repeat": info["repeat"],
            "fold": info["fold"],
            "threshold_strategy": strategy,
            "selected_threshold": threshold,
            **test_metrics,
            "cost_FN1_FP1_per100": (
                100.0
                * test_metrics["cost_FN1_FP1"]
                / n_test
            ),
            "cost_FN2_FP1_per100": (
                100.0
                * test_metrics["cost_FN2_FP1"]
                / n_test
            ),
            "cost_FN5_FP1_per100": (
                100.0
                * test_metrics["cost_FN5_FP1"]
                / n_test
            ),
            "cost_FN10_FP1_per100": (
                100.0
                * test_metrics["cost_FN10_FP1"]
                / n_test
            ),
        })


thresholds = pd.DataFrame(
    threshold_rows
)

outer_results = pd.DataFrame(
    outer_result_rows
)

thresholds.to_csv(
    OUT_DIR / "german_thresholds_by_outer_run.csv",
    index=False,
)

outer_results.to_csv(
    OUT_DIR / "german_threshold_outer_fold_results.csv",
    index=False,
)

print("Threshold selections:", len(thresholds))
print("Outer evaluations:", len(outer_results))

assert len(thresholds) == 25 * len(THRESHOLD_STRATEGIES)
assert len(outer_results) == 25 * len(THRESHOLD_STRATEGIES)


Threshold selections: 175
Outer evaluations: 175


## 7. German threshold-development summary

In [9]:

summary = (
    outer_results
    .groupby(
        "threshold_strategy",
        as_index=False,
    )
    .agg(
        Threshold_Mean=(
            "selected_threshold",
            "mean",
        ),
        Threshold_SD=(
            "selected_threshold",
            "std",
        ),
        Threshold_Min=(
            "selected_threshold",
            "min",
        ),
        Threshold_Max=(
            "selected_threshold",
            "max",
        ),
        Recall=(
            "recall_adverse",
            "mean",
        ),
        Precision=(
            "precision_adverse",
            "mean",
        ),
        F1=(
            "f1_adverse",
            "mean",
        ),
        Specificity=(
            "specificity",
            "mean",
        ),
        Balanced_Accuracy=(
            "balanced_accuracy",
            "mean",
        ),
        MCC=("mcc", "mean"),
        GMean=("gmean", "mean"),
        Cost_1_1=(
            "cost_FN1_FP1_per100",
            "mean",
        ),
        Cost_2_1=(
            "cost_FN2_FP1_per100",
            "mean",
        ),
        Cost_5_1=(
            "cost_FN5_FP1_per100",
            "mean",
        ),
        Cost_10_1=(
            "cost_FN10_FP1_per100",
            "mean",
        ),
    )
)

summary.to_csv(
    OUT_DIR / "german_threshold_development_summary.csv",
    index=False,
)

display(
    summary.sort_values(
        [
            "MCC",
            "Balanced_Accuracy",
            "F1",
        ],
        ascending=False,
    )
)


,threshold_strategy,Threshold_Mean,Threshold_SD,Threshold_Min,Threshold_Max,Recall,Precision,F1,Specificity,Balanced_Accuracy,MCC,GMean,Cost_1_1,Cost_2_1,Cost_5_1,Cost_10_1
5,MinCost_FN2_FP1,0.353756,0.040744,0.233351,0.414247,0.689360,0.553559,0.608808,0.758264,0.723812,0.425318,0.720324,26.34,35.72,63.86,110.76
3,MaxMCC,0.394149,0.071166,0.224304,0.545718,0.631011,0.580498,0.597511,0.796435,0.713723,0.421760,0.704078,25.18,36.16,69.10,124.00
2,MaxF1,0.319085,0.046009,0.233351,0.392378,0.731533,0.530248,0.609499,0.715753,0.723643,0.418687,0.720624,28.02,36.12,60.42,100.92
1,MaxBalancedAccuracy,0.311062,0.047862,0.233351,0.392378,0.739106,0.523064,0.607484,0.704365,0.721735,0.413939,0.718517,28.56,36.42,60.00,99.30
0,Fixed0.50,0.500000,0.000000,0.500000,0.500000,0.464135,0.646891,0.536208,0.891365,0.677750,0.395607,0.640875,23.74,39.86,88.22,168.82
6,MinCost_FN5_FP1,0.169435,0.029468,0.128020,0.224304,0.889801,0.416800,0.564666,0.462767,0.676284,0.338704,0.638594,41.04,44.38,54.40,71.10
4,MinCost_FN10_FP1,0.084766,0.020884,0.044097,0.128020,0.964936,0.354848,0.516922,0.244053,0.604494,0.248341,0.479001,54.06,55.12,58.30,63.60


## 8. Paired deltas against calibrated threshold 0.50

In [10]:

reference = (
    outer_results[
        outer_results[
            "threshold_strategy"
        ] == "Fixed0.50"
    ]
    [
        [
            "run_id",
            "recall_adverse",
            "precision_adverse",
            "f1_adverse",
            "balanced_accuracy",
            "mcc",
            "gmean",
            "cost_FN1_FP1_per100",
            "cost_FN2_FP1_per100",
            "cost_FN5_FP1_per100",
            "cost_FN10_FP1_per100",
        ]
    ]
    .copy()
)

reference = reference.rename(
    columns={
        c: "reference_" + c
        for c in reference.columns
        if c != "run_id"
    }
)

paired = outer_results.merge(
    reference,
    on="run_id",
    how="left",
    validate="many_to_one",
)

for metric in [
    "recall_adverse",
    "precision_adverse",
    "f1_adverse",
    "balanced_accuracy",
    "mcc",
    "gmean",
    "cost_FN1_FP1_per100",
    "cost_FN2_FP1_per100",
    "cost_FN5_FP1_per100",
    "cost_FN10_FP1_per100",
]:
    paired["delta_" + metric] = (
        paired[metric]
        - paired["reference_" + metric]
    )

paired.to_csv(
    OUT_DIR / "german_threshold_paired_deltas.csv",
    index=False,
)

delta_summary = (
    paired
    .groupby(
        "threshold_strategy",
        as_index=False,
    )
    .agg(
        Delta_Recall=(
            "delta_recall_adverse",
            "mean",
        ),
        Delta_Precision=(
            "delta_precision_adverse",
            "mean",
        ),
        Delta_F1=(
            "delta_f1_adverse",
            "mean",
        ),
        Delta_Balanced_Accuracy=(
            "delta_balanced_accuracy",
            "mean",
        ),
        Delta_MCC=(
            "delta_mcc",
            "mean",
        ),
        Delta_GMean=(
            "delta_gmean",
            "mean",
        ),
        Delta_Cost_1_1=(
            "delta_cost_FN1_FP1_per100",
            "mean",
        ),
        Delta_Cost_2_1=(
            "delta_cost_FN2_FP1_per100",
            "mean",
        ),
        Delta_Cost_5_1=(
            "delta_cost_FN5_FP1_per100",
            "mean",
        ),
        Delta_Cost_10_1=(
            "delta_cost_FN10_FP1_per100",
            "mean",
        ),
    )
)

delta_summary.to_csv(
    OUT_DIR / "german_threshold_delta_summary.csv",
    index=False,
)

display(delta_summary)


,threshold_strategy,Delta_Recall,Delta_Precision,Delta_F1,Delta_Balanced_Accuracy,Delta_MCC,Delta_GMean,Delta_Cost_1_1,Delta_Cost_2_1,Delta_Cost_5_1,Delta_Cost_10_1
0,Fixed0.50,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.00,0.00,0.00
1,MaxBalancedAccuracy,0.274971,-0.123827,0.071276,0.043985,0.018331,0.077642,4.82,-3.44,-28.22,-69.52
2,MaxF1,0.267398,-0.116642,0.073291,0.045893,0.023079,0.079749,4.28,-3.74,-27.80,-67.90
3,MaxMCC,0.166876,-0.066393,0.061303,0.035973,0.026152,0.063203,1.44,-3.70,-19.12,-44.82
4,MinCost_FN10_FP1,0.500801,-0.292043,-0.019286,-0.073256,-0.147266,-0.161874,30.32,15.26,-29.92,-105.22
5,MinCost_FN2_FP1,0.225225,-0.093331,0.072600,0.046062,0.029710,0.079449,2.60,-4.14,-24.36,-58.06
6,MinCost_FN5_FP1,0.425666,-0.230090,0.028458,-0.001466,-0.056904,-0.002281,17.30,4.52,-33.82,-97.72


## 9. Final checks

The default operational threshold rule will be frozen **after reviewing German results only**.

The cost-ratio strategies remain scenario analyses. A ratio such as FN:FP = 5:1 must not be described as a real monetary banking cost unless supported by domain-specific cost data.

After Step 11 is reviewed, the selected threshold rule will be replicated on Australian and Taiwan using their already-saved Step-10 probabilities and inner OOF predictions.


In [11]:

assert len(summary) == len(THRESHOLD_STRATEGIES)
assert len(delta_summary) == len(THRESHOLD_STRATEGIES)

assert outer_results["selected_threshold"].between(
    0.0,
    1.0,
).all()

assert outer_results["mcc"].between(
    -1.0,
    1.0,
).all()

for metric in [
    "recall_adverse",
    "precision_adverse",
    "f1_adverse",
    "balanced_accuracy",
    "gmean",
]:
    assert outer_results[metric].between(
        0.0,
        1.0,
    ).all()

configuration = {
    "stage": (
        "Objective 3 Step 11 - German cost-sensitive threshold development"
    ),
    "development_dataset": "German Credit",
    "frozen_upstream_framework": (
        "Group-aware Chi2 Top75 + balanced LR/RF/XGB + "
        "equal soft vote + sigmoid calibration"
    ),
    "threshold_strategies": THRESHOLD_STRATEGIES,
    "threshold_training_data": (
        "Cross-fitted sigmoid-calibrated OOF hybrid probabilities "
        "inside each outer-training partition"
    ),
    "cost_scenarios": {
        "2:1": "False negative cost = 2 x false positive cost",
        "5:1": "False negative cost = 5 x false positive cost",
        "10:1": "False negative cost = 10 x false positive cost",
    },
    "cost_interpretation": (
        "Sensitivity scenarios only; not actual monetary bank costs."
    ),
    "selection_rule": (
        "Default threshold strategy must be selected from German evidence only. "
        "Australian and Taiwan cannot influence threshold-rule selection."
    ),
}

with open(
    OUT_DIR / "step11_experiment_configuration.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        configuration,
        f,
        indent=4,
    )

manifest = sorted(
    [
        p.name
        for p in OUT_DIR.iterdir()
        if p.is_file()
    ]
)

pd.DataFrame(
    {"generated_file": manifest}
).to_csv(
    OUT_DIR / "step11_output_manifest.csv",
    index=False,
)

print("=" * 80)
print("STEP 11 COMPLETED SUCCESSFULLY")
print("=" * 80)
print("Output folder:", OUT_DIR)
print("\nMost important files:")
print(" - german_threshold_development_summary.csv")
print(" - german_threshold_delta_summary.csv")
print(" - german_threshold_outer_fold_results.csv")
print(" - german_thresholds_by_outer_run.csv")
print(" - german_threshold_crossfitted_calibrated_oof.csv")


STEP 11 COMPLETED SUCCESSFULLY
Output folder: D:\PHD\Research Paper writing\3rd Obj. paper\results\threshold_german_development

Most important files:
 - german_threshold_development_summary.csv
 - german_threshold_delta_summary.csv
 - german_threshold_outer_fold_results.csv
 - german_thresholds_by_outer_run.csv
 - german_threshold_crossfitted_calibrated_oof.csv
